Montgomery County Maryland Wine Sales Dashboard V2

Module imports

In [1]:
import pandas as pd
import numpy as np
import requests
import PyPDF2
import io
import re
import sqlite3
import time
import datetime
import plotly.graph_objects as go
import plotly.express as px
import warnings
import sys
import pickle
import os
import streamlit as st
from plotly.subplots import make_subplots
from difflib import SequenceMatcher
from datetime import datetime
from openpyxl import Workbook
from io import StringIO
import importlib

# Add utils path FIRST
sys.path.append('./utils')

# Import custom utilities
import fuzzy_supplier_matching as fuzzy
import enhanced_data_cleaning_utils as deu
import wine_classification_utils as wcu
import wine_review_matching_utils as wrmu
import inspect

# Reload fuzzy module after fixing indentation
importlib.reload(fuzzy)

warnings.filterwarnings('ignore')

File imports

In [2]:
# GitHub raw URL
base_url = "https://raw.githubusercontent.com/ac604605/Montgomery_County_Dashboard/main/"

print("Loading datasets from GitHub repository...")
print("=" * 60)

try:
    # Load standard datasets
    Distributors_Virginia_Three_Main = pd.read_csv(base_url + "data/Distributors_Virginia_Three_Main.csv")
    wine_producers = pd.read_csv(base_url + "data/wine_producers.csv")
    Warehouse_and_Retail_Sales = pd.read_csv(base_url + "data/Warehouse_and_Retail_Sales.csv")
    Wine_Review_Data = pd.read_csv(base_url + "data/winemag-data-130k-v2.csv/winemag-data-130k-v2.csv")
    
    # Load suppliers with data quality fix
    print("Loading and fixing supplier data structure...")
    correct_columns = ['License_ID', 'Trade Name', 'Address', 'City', 'State', 'Zip_Code', 'Report_Type']
    Suppliers_Fixed = pd.read_csv(
        base_url + "data/Suppliers_Importers_Retailers.csv",
        header=0,
        names=correct_columns,
        usecols=range(7),
        dtype={'License_ID': str, 'Zip_Code': str}
    )
    
    # Professional summaries
    datasets = [
        (Distributors_Virginia_Three_Main, "Virginia Distributors"),
        (wine_producers, "Wine Producers"),
        (Warehouse_and_Retail_Sales, "Sales Transactions"),
        (Wine_Review_Data, "Wine Reviews"),
        (Suppliers_Fixed, "Supplier Directory (Fixed)")  # Note the "Fixed" indicator
    ]
    
    for df, name in datasets:
        print(f"{name:<25} │ {df.shape[0]:>8,} rows × {df.shape[1]:>2} cols │ {df.memory_usage(deep=True).sum() / 1024**2:>6.1f} MB")
    
    # Quick validation for suppliers
    report_types = Suppliers_Fixed['Report_Type'].nunique()
    print(f"Supplier validation: {report_types} unique report types identified")
    
    print("=" * 60)
    print(f"Successfully loaded {len(datasets)} datasets with data quality fixes applied")
    
except Exception as e:
    print(f"Error loading data: {e}")

Loading datasets from GitHub repository...
Loading and fixing supplier data structure...
Virginia Distributors     │   14,279 rows ×  4 cols │    2.4 MB
Wine Producers            │    2,324 rows ×  7 cols │    0.9 MB
Sales Transactions        │  307,645 rows ×  9 cols │   87.5 MB
Wine Reviews              │  129,971 rows × 14 cols │  118.5 MB
Supplier Directory (Fixed) │   78,067 rows ×  7 cols │   31.2 MB
Supplier validation: 19 unique report types identified
Successfully loaded 5 datasets with data quality fixes applied


Now with everything loaded, I will begin data cleaning and refinement to suit the needs of this specific dashboard. 

After investigating the sales data, there are a few cleaning steps that need to take place. First will be removing all values that are not wine and beer items carried by distributors. Second will be ensuring item codes are numeric for easier processesing. Finally, some idividual values will be changed and anything that isn't wine or beer will be removed. I will also be removing the keg versions of wines and beer, as those would introduce greater scope that I do not wish to manage for a simple portfolio. 

In [20]:
# Create working copy for processing
print("Creating working copy of sales data...")
df_working = Warehouse_and_Retail_Sales.copy()
print(f"Working dataset: {df_working.shape[0]:,} rows × {df_working.shape[1]} columns")

# Run your enhanced data cleaning utilities
print("\nStarting data cleaning pipeline...")
df_clean, cleaning_report = deu.run_complete_item_code_standardization(
    df_working, 
    item_types_to_keep=['WINE', 'BEER']
)

# Show cleaning results
print(f"\nCleaning Results:")
print(f"   Original: {cleaning_report['original_shape']}")
print(f"   Cleaned:  {cleaning_report['final_shape']}")
print(f"   Retention: {cleaning_report['summary']['data_retention_pct']:.1f}%")

Creating working copy of sales data...
Working dataset: 307,645 rows × 9 columns

Starting data cleaning pipeline...
COMPLETE ITEM CODE STANDARDIZATION PIPELINE
Processing dataset with 307,645 rows and 9 columns
STEP 1: REMOVING MISSING SUPPLIER DATA
Original dataset shape: (307645, 9)
Rows with missing SUPPLIER: 167
Missing SUPPLIER percentage: 0.05%

 Cleaning complete!
New dataset shape: (307478, 9)
 Rows removed: 167
✓ Null SUPPLIER remaining: 0
Data reduction: 0.05%
STEP 2: ANALYZING NON-NUMERIC ITEM CODE PATTERNS
Total items analyzed: 307,478
Non-numeric item codes found: 7
Non-numeric percentage: 0.00%

 Analyzing patterns in non-numeric codes...
Pattern analysis:
Most common format: [numbers][letters]
Suffixes found:
  'A': 7 occurrences

 Sample non-numeric codes (showing up to 10):
97024A - GLUTENBERG PALE ALE - 16OZ CAN...
50029A - AVERY ELLIES BROWN 4/6 12OZ CAN...
300662A - DR STONERS SMOKY HERB WHISKEY 750ML...
348152A - VIRGINIA BLACK WHISKEY - 750ML...
81130A - CH LEOGN

I will also perform some joins and various cleaning of supporting data tables meant to make brand ownership rights more clear. 

In [21]:
# Run supplier enrichment with fuzzy matching
print("Starting supplier enrichment pipeline...")
df_enriched = fuzzy.run_supplier_enrichment(
    df_clean, 
    Suppliers_Fixed, 
    test_mode=False
)

# Show enrichment results
print(f"\nSupplier Enrichment Results:")
matched_suppliers = df_enriched[
    (df_enriched['SUPPLIER_MATCH_SCORE'] >= 0.8) & 
    (df_enriched['SUPPLIER_REPORT_TYPE'] == 'Wholesale Wine Distributors')
]
match_rate = len(matched_suppliers) / len(df_enriched) * 100
print(f"   Rows matched to distributors: {len(matched_suppliers):,}/{len(df_enriched):,} ({match_rate:.1f}%)")
print(f"   Unique distributors identified: {matched_suppliers['MATCHED_SUPPLIER_NAME'].nunique()}")

# Quick preview of top distributors
print(f"\nTop 5 Wholesale Wine Distributors by volume:")
print(matched_suppliers['SUPPLIER'].value_counts().head())

Starting supplier enrichment pipeline...
Running FULL enrichment...
Starting fuzzy supplier matching with threshold 0.8
Sales data: 230,053 rows
Wholesale Wine Distributors: 440 (filtered from 78,067 total suppliers)
Unique suppliers to match: 340
Processed 50/340 suppliers (11 matches, 3.3s)
Processed 100/340 suppliers (13 matches, 6.6s)
Processed 150/340 suppliers (17 matches, 9.9s)
Processed 200/340 suppliers (19 matches, 13.2s)
Processed 250/340 suppliers (23 matches, 16.5s)
Processed 300/340 suppliers (24 matches, 19.8s)
Applying matches to all rows...

FUZZY MATCHING RESULTS:
Unique suppliers matched: 24/340
Total rows matched: 48,657/230,053 (21.2%)
Processing time: 27.8 seconds
Average time per supplier: 0.082s

Supplier Enrichment Results:
   Rows matched to distributors: 48,657/230,053 (21.2%)
   Unique distributors identified: 24

Top 5 Wholesale Wine Distributors by volume:
SUPPLIER
REPUBLIC NATIONAL DISTRIBUTING CO    17598
MONSIEUR TOUTON SELECTION            10326
DIONYS

After reviewing the wine review data, it appears this will serve as an adequate database to gather missing country data for our sales table. For the sake of simplicity, I will combine all columns from this table to matching wines in our sales table. Columns we do not need can be filtered out later. 

full_results, sales_map, review_map = wrmu.run_wine_review_matching(
    df_clean_Warehouse_and_Retail_Sales, Wine_Review_Data, threshold=0.6, test_mode=False
)

In [26]:
# Save as pickle (recommended for data analysis)
#full_results.to_pickle('wine_sales_with_reviews_FINAL.pkl')

# To load later:
df = pd.read_pickle('wine_sales_with_reviews_FINAL.pkl')

In [27]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
#df=df_loaded
#df=df_final
df=df

#uncomment below lines to change report
#df.head(25)
df.info()
#df.describe()
#df[df['final_variety'] == ''].head()
#pd.set_option('display.max_rows', None)
#df['ITEM DESCRIPTION'].value_counts()
#deu.investigate_missing_values_manually(df)

<class 'pandas.core.frame.DataFrame'>
Index: 187640 entries, 0 to 307637
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   YEAR                 187640 non-null  int64  
 1   MONTH                187640 non-null  int64  
 2   SUPPLIER             187640 non-null  object 
 3   ITEM CODE            187640 non-null  int64  
 4   ITEM DESCRIPTION     187640 non-null  object 
 5   ITEM TYPE            187640 non-null  object 
 6   RETAIL SALES         187640 non-null  float64
 7   RETAIL TRANSFERS     187640 non-null  float64
 8   WAREHOUSE SALES      187640 non-null  float64
 9   review_title         187640 non-null  object 
 10  review_country       187632 non-null  object 
 11  review_variety       187640 non-null  object 
 12  review_points        187640 non-null  object 
 13  review_price         182737 non-null  object 
 14  review_description   187640 non-null  object 
 15  review_province      1

In [6]:
# After wine review matching, add classification
print("Starting comprehensive wine classification...")
df_classified, variety_counts, country_counts = wcu.run_enhanced_wine_classification(df)

# Quick summary
wcu.quick_enhanced_summary(df_classified)

# Final enriched dataset
df_complete = df_classified
print(f"Complete pipeline finished! Dataset: {df_complete.shape}")

Starting comprehensive wine classification...
=== ENHANCED WINE CLASSIFICATION SYSTEM WITH COUNTRY DETECTION ===
Processing 187,640 records...

Step 1: Applying sparkling classification...
Step 2: Applying variety consolidation...
Step 3: Applying text extraction for varieties...
Wines needing variety extraction: 106,286
  Processed 5,000 variety extractions...
  Processed 10,000 variety extractions...
  Processed 15,000 variety extractions...
  Processed 20,000 variety extractions...
  Processed 25,000 variety extractions...
  Processed 30,000 variety extractions...
  Processed 35,000 variety extractions...
  Processed 40,000 variety extractions...
  Processed 45,000 variety extractions...
  Processed 50,000 variety extractions...
  Processed 55,000 variety extractions...
  Processed 60,000 variety extractions...
  Processed 65,000 variety extractions...
  Processed 70,000 variety extractions...
  Processed 75,000 variety extractions...
  Processed 80,000 variety extractions...
  Proc

In [7]:
# Apply wine color classification to your final dataset
df_classified_with_color['wine_color'] = df_classified['final_variety'].apply(classify_wine_color)

print("Wine color distribution:")
print(df_complete['wine_color'].value_counts())

NameError: name 'classify_wine_color' is not defined

In [3]:
# Save as pickle (recommended for data analysis)
df_classified_with_color.to_pickle('wine_data_fully_classified.pkl')

# To load later:
#df = pd.read_pickle('wine_data_fully_classified.pkl')

In [10]:
# Get the full series as a readable format
variety_counts = df['final_variety'].value_counts()
for variety, count in variety_counts.items():
    print(f"{variety}: {count}")

: 25170
Red Blend: 21392
Cabernet Sauvignon: 18486
Chardonnay: 16700
Pinot Noir: 13767
Sauvignon Blanc: 10217
Rosé: 7852
Pinot Grigio: 7735
Merlot: 7084
White Blend: 7084
Malbec: 4545
Moscato: 4166
Syrah: 3957
Riesling: 3673
Zinfandel: 3190
Tempranillo: 2427
Sake: 2400
Sangiovese: 2079
Port: 1695
Prosecco: 1495
Glera: 1343
Sparkling Blend: 1271
Champagne Blend: 1167
Nebbiolo: 1092
Cabernet Franc: 733
Sherry: 724
Concord: 664
Montepulciano: 661
White Zinfandel: 655
Chianti: 647
Gamay: 620
Barbera: 567
Albariño: 561
Fruit Wine: 553
Viognier: 526
Shiraz: 509
Garnacha: 498
Portuguese White: 489
Gewürztraminer: 483
Portuguese Red: 476
Chenin Blanc: 432
Tempranillo Blend: 408
Nero d'Avola: 362
Petite Sirah: 349
Monastrell: 310
Garganega: 292
Verdejo: 278
Catarratto: 275
Viura: 255
Amarone: 179
Primitivo: 175
Vinho Verde: 173
Pinotage: 173
Corvina, Rondinella, Molinara: 159
Saperavi: 140
Grillo: 136
Grüner Veltliner: 135
Nerello Mascalese: 123
Cortese: 118
Torrontés: 109
Lambrusco: 107
Carric